# Superstore Sales — EDA
## 1. Импорты и настройки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook"

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_columns", None)


## 2. Загрузка и первичный осмотр данных

In [ ]:
# ---------- Загрузка ----------
df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin-1")
print("Shape:", df.shape)
df.head()


## 3. Очистка данных

In [ ]:
# 1. Дубликаты
print("Дубликатов до:", df.duplicated().sum())
df = df.drop_duplicates()
print("Дубликатов после:", df.duplicated().sum())

# 2. Пропуски
print("\nПропуски:\n", df.isnull().sum()[df.isnull().sum() > 0])

# 3. Типы
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format="%m/%d/%Y")

# 4. Проверка числовых полей
df[["Sales", "Quantity", "Discount", "Profit"]] = df[
    ["Sales", "Quantity", "Discount", "Profit"]
].apply(pd.to_numeric, errors="coerce")

# 5. Отрицательные/нулевые продажи — аномалии
print("Продажи <= 0:", (df["Sales"] <= 0).sum())
df = df[df["Sales"] > 0]

# 6. Выбросы через IQR (для Sales и Profit)
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)

print("Выбросов Sales:", iqr_outliers(df["Sales"]).sum())
print("Выбросов Profit:", iqr_outliers(df["Profit"]).sum())
# НЕ удаляем — в бизнесе это реальные крупные сделки, но фиксируем в отчёте

## 4. Feature Engineering

In [ ]:
df["Order Year"]      = df["Order Date"].dt.year
df["Order Month"]     = df["Order Date"].dt.month
df["Order Month Name"]= df["Order Date"].dt.strftime("%b")
df["Order Day of Week"]= df["Order Date"].dt.day_name()
df["Order Quarter"]   = df["Order Date"].dt.quarter

# Сезон
def season(m):
    if m in [12, 1, 2]: return "Winter"
    if m in [3, 4, 5]:  return "Spring"
    if m in [6, 7, 8]:  return "Summer"
    return "Autumn"
df["Season"] = df["Order Month"].apply(season)

# Дни до отгрузки (Ship lag)
df["Ship Lag Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Маржа
df["Profit Margin"] = df["Profit"] / df["Sales"]

# Категория скидки
df["Discount Band"] = pd.cut(
    df["Discount"],
    bins=[-0.001, 0, 0.2, 0.4, 1.0],
    labels=["No Discount", "Low", "Medium", "High"]
)
df.head()

## 5. Анализ через groupby и pivot_table

In [ ]:
# Топ регионов по прибыли
region_profit = (df.groupby("Region")
                   .agg(Sales=("Sales","sum"),
                        Profit=("Profit","sum"),
                        Orders=("Order ID","nunique"))
                   .sort_values("Profit", ascending=False))
print(region_profit)

# Прибыльность по категориям
cat_stats = df.groupby(["Category","Sub-Category"]).agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Avg_Discount=("Discount","mean")
).round(2).sort_values("Profit", ascending=False)
print(cat_stats.head(10))

# Pivot: Месяц × Категория
pivot = pd.pivot_table(
    df, values="Sales", index="Order Month",
    columns="Category", aggfunc="sum", fill_value=0
)
print(pivot)

# Pivot: Скидка → Прибыль
pivot_disc = pd.pivot_table(
    df, values="Profit", index="Discount Band",
    columns="Category", aggfunc="mean"
)
print(pivot_disc)

## 6. Статичные визуализации (matplotlib / seaborn)

In [ ]:
import os
os.makedirs("reports/figures", exist_ok=True)

# 1. Динамика продаж и прибыли по месяцам
monthly = df.groupby(["Order Year","Order Month"]).agg(
    Sales=("Sales","sum"), Profit=("Profit","sum")
).reset_index()
monthly["Date"] = pd.to_datetime(
    monthly["Order Year"].astype(str) + "-" + monthly["Order Month"].astype(str) + "-01"
)

fig, ax1 = plt.subplots(figsize=(14,6))
ax1.plot(monthly["Date"], monthly["Sales"], color="tab:blue", label="Sales")
ax1.set_ylabel("Sales", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(monthly["Date"], monthly["Profit"], color="tab:red", label="Profit")
ax2.set_ylabel("Profit", color="tab:red")
plt.title("Monthly Sales & Profit")
plt.tight_layout()
plt.savefig("../reports/figures/monthly_sales_profit.png", dpi=120)
plt.show()

# 2. Топ-10 подкатегорий по прибыли
top_sub = (df.groupby("Sub-Category")["Profit"].sum()
             .sort_values(ascending=False).head(10))
plt.figure(figsize=(10,6))
sns.barplot(x=top_sub.values, y=top_sub.index, palette="viridis")
plt.title("Top-10 Sub-Categories by Profit")
plt.tight_layout()
plt.savefig("../reports/figures/top_subcategories.png", dpi=120)
plt.show()

# 3. Зависимость Profit от Discount
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x="Discount", y="Profit",
                hue="Category", alpha=0.5)
plt.axhline(0, color="black", linestyle="--")
plt.title("Profit vs Discount")
plt.tight_layout()
plt.savefig("../reports/figures/profit_vs_discount.png", dpi=120)
plt.show()

# 4. Корреляционная матрица
plt.figure(figsize=(8,6))
sns.heatmap(df[["Sales","Quantity","Discount","Profit","Ship Lag Days"]].corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig("../reports/figures/correlation.png", dpi=120)
plt.show()

## 7. Интерактивные визуализации (Plotly)

In [ ]:
# 1. Продажи по штатам (choropleth)
state_sales = df.groupby("State")["Sales"].sum().reset_index()
fig = px.choropleth(
    state_sales, locations="State", locationmode="USA-states",
    color="Sales", scope="usa", color_continuous_scale="Blues",
    title="Sales by State"
)
fig.show()

# 2. Treemap: Category → Sub-Category
fig = px.treemap(
    df, path=["Category","Sub-Category"],
    values="Profit", color="Profit",
    color_continuous_scale="RdYlGn",
    title="Profit Treemap by Category"
)
fig.show()

# 3. Sunburst: Region → Segment → Category
fig = px.sunburst(
    df, path=["Region","Segment","Category"],
    values="Sales", title="Sales Breakdown"
)
fig.show()

# 4. Динамика по месяцам с селектором года
fig = px.line(
    monthly, x="Date", y="Sales", color="Order Year",
    markers=True, title="Sales Dynamics"
)
fig.show()

# 5. Box-plot прибыли по скидкам
fig = px.box(df, x="Discount Band", y="Profit", color="Category",
             title="Profit Distribution by Discount Band")
fig.show()

## 8. Бизнес-выводы в README